# ARLE Code Completion Ranking Model Training
# ============================================
#
# **CRITICAL**: This model trains for RANKING completions, NOT language detection!
#
# Architecture: 2-layer Bi-directional LSTM + Attention
# Task: Learn to score completion quality (pairwise ranking loss)
# Target: 20MB INT8 quantized model
#
# Run on Google Colab with T4 GPU (free tier compatible)
# Estimated time: 3-4 hours for 51 languages

## Step 0: GPU Check & Setup

In [ ]:
import torch
import os

print("=" * 60)
print("RESOURCE CHECK")
print("=" * 60)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! This will be VERY slow. Enable GPU in Runtime → Change runtime type")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Step 1: Install Dependencies

In [ ]:
!pip install -q datasets tokenizers torch onnx onnxruntime tqdm psutil huggingface_hub pyyaml

## Step 2: Configuration

In [ ]:
import yaml

# Load configuration (or use defaults)
CONFIG = {
    'model': {
        'embed_dim': 128,
        'lstm_hidden': 128,
        'lstm_layers': 2,
        'attention_dim': 64,
        'dropout': 0.1,
        'vocab_size': 8000,
        'max_seq_len': 128,
    },
    'training': {
        'batch_size': 64,
        'epochs': 5,
        'learning_rate': 0.001,
        'samples_per_lang': 5000,
        'train_split': 0.95,
        'margin': 0.5,  # for ranking loss
    },
    'languages': {
        'tier_1': ['python', 'javascript', 'typescript', 'java', 'go', 'rust', 'cpp', 'c', 'php', 'ruby'],
        'tier_2': ['swift', 'kotlin', 'scala', 'csharp', 'dart', 'lua', 'r', 'julia', 'perl', 'shell'],
        'tier_3': ['haskell', 'clojure', 'elixir', 'erlang', 'groovy', 'powershell', 'objective-c',
                   'fsharp', 'ocaml', 'fortran', 'cobol', 'ada', 'prolog', 'common-lisp', 'scheme',
                   'racket', 'emacs-lisp', 'assembly', 'vhdl', 'verilog', 'zig', 'nim', 'crystal'],
        'tier_4': ['json', 'yaml', 'toml', 'xml', 'markdown', 'dockerfile', 'sql', 'html'],
    }
}

# Flatten languages
ALL_LANGUAGES = CONFIG['languages']['tier_1'] + CONFIG['languages']['tier_2'] + \
                CONFIG['languages']['tier_3'] + CONFIG['languages']['tier_4']

print(f"Total languages: {len(ALL_LANGUAGES)}")
print(f"Model config: {CONFIG['model']}")
print(f"Training config: {CONFIG['training']}")

## Step 3: HuggingFace Authentication

In [ ]:
from huggingface_hub import login
from getpass import getpass

# Option 1: Paste your token directly (not recommended for shared notebooks)
# HF_TOKEN = "hf_..."

# Option 2: Prompt for token (recommended)
HF_TOKEN = getpass("Enter your HuggingFace token: ")

login(token=HF_TOKEN)
print("✅ Logged in to HuggingFace")

## Step 4: Load and Prepare Data

We'll load code from The Stack and create **pairwise ranking samples**:
- **Context**: Code before cursor
- **Positive**: Actual next line (ground truth)
- **Negative**: Random/wrong completion

In [ ]:
from datasets import load_dataset
from tqdm import tqdm
import random
import re

# Full language mapping for The Stack (dedup) - 52 languages
# Format: internal_name -> the-stack language identifier
LANG_MAP = {
    # Tier 1 - Core (10)
    'python': 'python',
    'javascript': 'javascript',
    'typescript': 'typescript',
    'java': 'java',
    'go': 'go',
    'rust': 'rust',
    'cpp': 'c++',
    'c': 'c',
    'php': 'php',
    'ruby': 'ruby',
    # Tier 2 - Important (10)
    'swift': 'swift',
    'kotlin': 'kotlin',
    'scala': 'scala',
    'csharp': 'c-sharp',
    'dart': 'dart',
    'lua': 'lua',
    'r': 'r',
    'julia': 'julia',
    'perl': 'perl',
    'shell': 'shell',
    # Tier 3 - Specialized (22)
    'haskell': 'haskell',
    'clojure': 'clojure',
    'elixir': 'elixir',
    'erlang': 'erlang',
    'groovy': 'groovy',
    'powershell': 'powershell',
    'objectivec': 'objective-c',
    'fsharp': 'f-sharp',
    'ocaml': 'ocaml',
    'fortran': 'fortran',
    'cobol': 'cobol',
    'ada': 'ada',
    'prolog': 'prolog',
    'commonlisp': 'common-lisp',
    'scheme': 'scheme',
    'racket': 'racket',
    'assembly': 'assembly',
    'vhdl': 'vhdl',
    'verilog': 'verilog',
    'zig': 'zig',
    'nim': 'nim',
    'crystal': 'crystal',
    # Tier 4 - Data/Markup (9)
    'json': 'json',
    'yaml': 'yaml',
    'toml': 'toml',
    'xml': 'xml',
    'markdown': 'markdown',
    'dockerfile': 'dockerfile',
    'sql': 'sql',
    'html': 'html',
    'css': 'css',
}

def create_completion_pairs(code, max_context=10):
    \"\"\"Split code into (context, completion) pairs.\"\"\"
    lines = code.split('\\n')
    pairs = []
    
    for i in range(1, len(lines)):
        if len(lines[i].strip()) < 5:
            continue
        start_idx = max(0, i - max_context)
        context = '\\n'.join(lines[start_idx:i])
        completion = lines[i].strip()
        
        if len(context.strip()) > 0 and len(completion) > 0:
            pairs.append((context, completion))
    
    return pairs

def load_language_data(lang_key, lang_id, n_samples=5000):
    \"\"\"Load data for one language from The Stack (dedup).\"\"\"
    print(f\"Loading {lang_key} ({lang_id})...\")
    
    try:
        # Use The Stack v1 (dedup) - open access, no gating
        ds = load_dataset(
            \"bigcode/the-stack-dedup\",
            data_dir=f\"data/{lang_id}\",
            split=\"train\",
            streaming=True,
        )
        
        all_pairs = []
        all_completions = []
        
        for item in ds:
            if len(all_pairs) >= n_samples * 2:
                break
                
            content = item.get(\"content\", \"\")
            if not content or len(content) < 100 or len(content) > 10000:
                continue
            
            pairs = create_completion_pairs(content)
            all_pairs.extend(pairs)
            all_completions.extend([p[1] for p in pairs])
        
        if len(all_pairs) == 0:
            print(f\"  ✗ {lang_key}: no data found\")
            return []
        
        # Create triplets (context, positive, negative)
        triplets = []
        for context, positive in all_pairs[:n_samples]:
            negative = random.choice(all_completions)
            while negative == positive and len(all_completions) > 1:
                negative = random.choice(all_completions)
            
            triplets.append({
                'context': context,
                'positive': positive,
                'negative': negative,
                'language': lang_key
            })
        
        print(f\"  ✓ {lang_key}: {len(triplets)} triplets\")
        return triplets
        
    except Exception as e:
        print(f\"  ✗ {lang_key}: {e}\")
        return []

# Load data for all languages
print(\"\\nLoading training data...\")
print(\"=\" * 60)

all_training_data = []
samples_per_lang = CONFIG['training']['samples_per_lang']

for lang in ALL_LANGUAGES:
    if lang in LANG_MAP:
        lang_id = LANG_MAP[lang]
        triplets = load_language_data(lang, lang_id, samples_per_lang)
        all_training_data.extend(triplets)

print(f\"\\n✓ Total training samples: {len(all_training_data)}\")

# Shuffle
random.shuffle(all_training_data)

## Step 5: Build Tokenizer

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, processors
import json

print("Building BPE tokenizer...")

# Create BPE tokenizer
tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

# Special tokens
special_tokens = ["<PAD>", "<UNK>", "<BOS>", "<EOS>", "<SEP>"]

# Train tokenizer
trainer = trainers.BpeTrainer(
    vocab_size=CONFIG['model']['vocab_size'],
    special_tokens=special_tokens,
    show_progress=True,
    min_frequency=3,
)

# Collect all text for training
training_texts = []
for item in tqdm(all_training_data, desc="Preparing tokenizer corpus"):
    training_texts.append(item['context'])
    training_texts.append(item['positive'])
    training_texts.append(item['negative'])

# Train
tokenizer.train_from_iterator(training_texts, trainer)

# Add post-processor
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)

# Save tokenizer
tokenizer_path = "arle_tokenizer.json"
tokenizer.save(tokenizer_path)
print(f"✓ Tokenizer saved: {tokenizer_path}")
print(f"✓ Vocab size: {tokenizer.get_vocab_size()}")

# Test
test_code = "def hello_world():\n    print('Hello')"
encoded = tokenizer.encode(test_code)
print(f"\nTest encoding: {encoded.tokens[:10]}")

## Step 6: Create PyTorch Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class RankingDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.pad_id = tokenizer.token_to_id("<PAD>")
        self.sep_id = tokenizer.token_to_id("<SEP>")
    
    def __len__(self):
        return len(self.data)
    
    def encode_pair(self, context, completion):
        """Encode context + completion with <SEP> separator."""
        ctx_ids = self.tokenizer.encode(context).ids
        comp_ids = self.tokenizer.encode(completion).ids
        
        # Combine: context <SEP> completion
        ids = ctx_ids + [self.sep_id] + comp_ids
        
        # Truncate or pad
        if len(ids) > self.max_length:
            ids = ids[:self.max_length]
        else:
            ids = ids + [self.pad_id] * (self.max_length - len(ids))
        
        return ids
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Encode positive pair
        positive_ids = self.encode_pair(item['context'], item['positive'])
        
        # Encode negative pair
        negative_ids = self.encode_pair(item['context'], item['negative'])
        
        return {
            'positive': torch.tensor(positive_ids, dtype=torch.long),
            'negative': torch.tensor(negative_ids, dtype=torch.long),
        }

# Split data
train_size = int(CONFIG['training']['train_split'] * len(all_training_data))
train_data = all_training_data[:train_size]
val_data = all_training_data[train_size:]

print(f"Train samples: {len(train_data)}")
print(f"Val samples: {len(val_data)}")

# Create datasets
train_dataset = RankingDataset(train_data, tokenizer, CONFIG['model']['max_seq_len'])
val_dataset = RankingDataset(val_data, tokenizer, CONFIG['model']['max_seq_len'])

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['training']['batch_size'],
    shuffle=True,
    num_workers=2,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['training']['batch_size'],
    shuffle=False,
    num_workers=2,
)

print(f"✓ Dataloaders created")

## Step 7: Define Model Architecture

**2-Layer Bi-directional LSTM + Attention for Ranking**

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim, attention_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, attention_dim),
            nn.Tanh(),
            nn.Linear(attention_dim, 1)
        )
    
    def forward(self, lstm_output, mask=None):
        # lstm_output: (batch, seq_len, hidden_dim)
        # Compute attention weights
        attn_weights = self.attention(lstm_output).squeeze(-1)  # (batch, seq_len)
        
        # Apply mask (ignore padding)
        if mask is not None:
            attn_weights = attn_weights.masked_fill(mask == 0, -1e9)
        
        # Softmax
        attn_weights = F.softmax(attn_weights, dim=1)
        
        # Weighted sum
        context = torch.bmm(attn_weights.unsqueeze(1), lstm_output).squeeze(1)
        return context

class ArleRankingModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        
        vocab_size = config['vocab_size']
        embed_dim = config['embed_dim']
        lstm_hidden = config['lstm_hidden']
        lstm_layers = config['lstm_layers']
        attention_dim = config['attention_dim']
        dropout = config['dropout']
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Bi-directional LSTM
        self.lstm = nn.LSTM(
            embed_dim,
            lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0,
        )
        
        # Attention (operates on bidirectional output)
        self.attention = AttentionLayer(lstm_hidden * 2, attention_dim)
        
        # Output: score layer (single score per sequence)
        self.score_layer = nn.Sequential(
            nn.Linear(lstm_hidden * 2, lstm_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, 1)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() > 1:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
    
    def forward(self, input_ids):
        # input_ids: (batch, seq_len)
        
        # Embedding
        embeds = self.embedding(input_ids)  # (batch, seq_len, embed_dim)
        
        # LSTM
        lstm_out, _ = self.lstm(embeds)  # (batch, seq_len, hidden*2)
        
        # Attention pooling
        mask = (input_ids != 0).float()  # Mask for padding
        context = self.attention(lstm_out, mask)  # (batch, hidden*2)
        
        # Score
        score = self.score_layer(context).squeeze(-1)  # (batch,)
        
        return score

# Create model
model = ArleRankingModel(CONFIG['model']).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "=" * 60)
print("MODEL ARCHITECTURE")
print("=" * 60)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Estimated size (FP32): {total_params * 4 / 1e6:.1f} MB")
print(f"Estimated size (INT8): {total_params / 1e6:.1f} MB")
print("=" * 60)

print(model)

## Step 8: Training Loop

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

# Loss function: Margin Ranking Loss
# We want: score(positive) > score(negative) + margin
margin = CONFIG['training']['margin']
criterion = nn.MarginRankingLoss(margin=margin)

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=CONFIG['training']['learning_rate'],
    weight_decay=0.01
)

# Scheduler
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['training']['epochs'] * len(train_loader)
)

def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc="Training")
    for batch in pbar:
        positive_ids = batch['positive'].to(device)
        negative_ids = batch['negative'].to(device)
        
        # Forward pass
        positive_scores = model(positive_ids)
        negative_scores = model(negative_ids)
        
        # Target: positive should score higher (target = 1)
        target = torch.ones(positive_scores.size(0), device=device)
        
        # Loss
        loss = criterion(positive_scores, negative_scores, target)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        scheduler.step()
        
        # Metrics
        total_loss += loss.item()
        correct += (positive_scores > negative_scores).sum().item()
        total += positive_scores.size(0)
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * correct / total
    
    return avg_loss, accuracy

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating"):
            positive_ids = batch['positive'].to(device)
            negative_ids = batch['negative'].to(device)
            
            positive_scores = model(positive_ids)
            negative_scores = model(negative_ids)
            
            target = torch.ones(positive_scores.size(0), device=device)
            loss = criterion(positive_scores, negative_scores, target)
            
            total_loss += loss.item()
            correct += (positive_scores > negative_scores).sum().item()
            total += positive_scores.size(0)
    
    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * correct / total
    
    return avg_loss, accuracy

# Training
print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

best_val_acc = 0
training_start = time.time()

for epoch in range(1, CONFIG['training']['epochs'] + 1):
    print(f"\nEpoch {epoch}/{CONFIG['training']['epochs']}")
    print("-" * 60)
    
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, scheduler, device
    )
    
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "arle_model_best.pth")
        print(f"✓ Saved checkpoint (val_acc: {val_acc:.2f}%)")

elapsed = time.time() - training_start
print(f"\n✓ Training completed in {elapsed / 60:.1f} minutes")
print(f"✓ Best validation accuracy: {best_val_acc:.2f}%")

## Step 9: Export to ONNX with INT8 Quantization

In [ ]:
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

# Load best model
model.load_state_dict(torch.load("arle_model_best.pth"))
model.eval()

print("Exporting to ONNX...")

# Dummy input
dummy_input = torch.randint(
    0, CONFIG['model']['vocab_size'],
    (1, CONFIG['model']['max_seq_len']),
    device=device
)

# Export to ONNX (FP32)
onnx_path_fp32 = "arle_model_fp32.onnx"\n
torch.onnx.export(
    model,
    dummy_input,
    onnx_path_fp32,
    input_names=["input_ids"],
    output_names=["score"],
    dynamic_axes={
        "input_ids": {0: "batch_size"},
        "score": {0: "batch_size"},
    },
    opset_version=14,
    do_constant_folding=True,
)

fp32_size = os.path.getsize(onnx_path_fp32) / 1024 / 1024
print(f"✓ ONNX FP32 exported: {onnx_path_fp32} ({fp32_size:.2f} MB)")

# Quantize to INT8
print("\nQuantizing to INT8...")
onnx_path_int8 = "arle_model.onnx"\n
quantize_dynamic(
    onnx_path_fp32,
    onnx_path_int8,
    weight_type=QuantType.QInt8,
)

int8_size = os.path.getsize(onnx_path_int8) / 1024 / 1024
print(f"✓ ONNX INT8 exported: {onnx_path_int8} ({int8_size:.2f} MB)")
print(f"✓ Compression: {fp32_size / int8_size:.1f}x smaller")

# Verify model
onnx_model = onnx.load(onnx_path_int8)
onnx.checker.check_model(onnx_model)
print("✓ ONNX model verified")

## Step 10: Test Exported Model

In [ ]:
import onnxruntime as ort
import numpy as np

print("Testing ONNX model...")

# Load ONNX model
session = ort.InferenceSession(onnx_path_int8)

# Test cases
test_cases = [
    {
        'context': 'def fibonacci(n):\n    if n <= 1:\n        return n',
        'good': '    return fibonacci(n-1) + fibonacci(n-2)',
        'bad': 'print("hello world")'
    },
    {
        'context': 'function add(a, b) {',
        'good': '    return a + b;',
        'bad': 'def hello(): pass'
    },
]

def encode_for_inference(context, completion):
    ctx_ids = tokenizer.encode(context).ids
    comp_ids = tokenizer.encode(completion).ids
    sep_id = tokenizer.token_to_id("<SEP>")
    pad_id = tokenizer.token_to_id("<PAD>")
    
    ids = ctx_ids + [sep_id] + comp_ids
    
    if len(ids) > CONFIG['model']['max_seq_len']:
        ids = ids[:CONFIG['model']['max_seq_len']]
    else:
        ids = ids + [pad_id] * (CONFIG['model']['max_seq_len'] - len(ids))
    
    return np.array([ids], dtype=np.int64)

print("\n" + "=" * 60)
print("MODEL TEST")
print("=" * 60)

for i, test in enumerate(test_cases, 1):
    print(f"\nTest {i}:")
    print(f"Context: {test['context'][:50]}...")
    
    # Score good completion
    good_input = encode_for_inference(test['context'], test['good'])
    good_score = session.run(None, {'input_ids': good_input})[0][0]
    
    # Score bad completion
    bad_input = encode_for_inference(test['context'], test['bad'])
    bad_score = session.run(None, {'input_ids': bad_input})[0][0]
    
    print(f"  Good: {test['good'][:40]}... → score={good_score:.3f}")
    print(f"  Bad:  {test['bad'][:40]}... → score={bad_score:.3f}")
    
    if good_score > bad_score:
        print("  ✓ PASS: Good completion scored higher")
    else:
        print("  ✗ FAIL: Bad completion scored higher!")

print("\n" + "=" * 60)

## Step 11: Download Files

Download these files:
1. `arle_model.onnx` - Quantized INT8 model\n2. `arle_tokenizer.json` - BPE tokenizer\n
Copy to `arlecchino/assets/` directory.

In [ ]:
print("\n" + "=" * 60)
print("FILES TO DOWNLOAD")
print("=" * 60)

files = [
    "arle_model.onnx",
    "arle_model_fp32.onnx",
    "arle_tokenizer.json",
    "arle_model_best.pth",
]

for f in files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024 / 1024
        print(f"  ✓ {f}: {size:.2f} MB")
    else:
        print(f"  ✗ {f}: NOT FOUND")

print("\nUse Files panel (left sidebar) → Right-click → Download")
print("Or run: from google.colab import files; files.download('arle_model.onnx')")

# Optional: Auto-download
# from google.colab import files
# files.download('arle_model.onnx')
# files.download('arle_tokenizer.json')